In [11]:
import os
import subprocess
import pandas as pd
import numpy as np
from datetime import timedelta
import georinex as gr
from datetime import datetime, timedelta

# -----------------------------
# ПАРАМЕТРЫ (настройте под себя)
# -----------------------------
ROVER_RNX = "/Users/sergeidolin/ART/RTKLIB/INPUT_DATA/P40_14072025/GEOP195K.obs"          # ваш RINEX-файл смартфона
BASE_FILE  = "/Users/sergeidolin/ART/RTKLIB/INPUT_DATA/P40_14072025/NSKN141.obs"           # RINEX базовой станции (NSKN)
NAV_FILE   = "/Users/sergeidolin/ART/RTKLIB/INPUT_DATA/P40_14072025/BRDC00WRD_R_20251950000_01D_MN.rnx"               # навигационный файл
SP3_FILE   = "/Users/sergeidolin/ART/RTKLIB/INPUT_DATA/P40_14072025/COD0OPSRAP_20251950000_01D_05M_ORB.SP3"                # точные эфемериды (если PPP — не нужен)
CLK_FILE   = "/Users/sergeidolin/ART/RTKLIB/INPUT_DATA/P40_14072025/COD0OPSRAP_20251950000_01D_30S_CLK.CLK"                # часы спутников (если PPP — не нужен)

CONFIG_FILE = "ppp-static.conf"           # ваш конфигурационный файл RTKLIB
# CONFIG_FILE = "ppp-staticnophase.conf"           # ваш конфигурационный файл RTKLIB

# CONFIG_FILE = "opts2.conf"           # ваш конфигурационный файл RTKLIB

# CONFIG_FILE = "opts1.conf"     
# CONFIG_FILE = "ppp-staticnophase.conf"           # ваш конфигурационный файл RTKLIB

# OUTPUT_DIR  = "solutions_nophase"             # папка для решений
# OUTPUT_DIR  = "solutions_nophase_gps"       
# OUTPUT_DIR  = "solutions"      
# OUTPUT_DIR  = "solutions_gps_AR"   
# OUTPUT_DIR  = "solutions_gps_NOAR"
# OUTPUT_DIR  = "solutions_all_NOAR"
OUTPUT_DIR  = "solutions_all_AR"
 
# OUTPUT_DIR  = "solutions_nophase_gps_rel"      
# OUTPUT_DIR  = "solutions_gps_rel"         # папка для решений
# OUTPUT_DIR  = "solutions_nophase_gps"       
# OUTPUT_DIR  = "solutions"      
# OUTPUT_DIR  = "solutions_gps"  
# OUTPUT_DIR  = "solutions_all_rel"  
# OUTPUT_DIR  = "solutions_nophase_all_rel" 
SEGMENT_MIN = 150                      # длительность сегмента в минутах
START_TIME  = "2025/07/14 10:12:32"  # Укажите начало сеанса 25 07 14 10 12 31.9999398
DURATION_H  = 20

# -----------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

start = datetime.strptime(START_TIME, "%Y/%m/%d %H:%M:%S")
end = start + timedelta(hours=DURATION_H)
current = start
segment_idx = 0

while current < end:
    seg_end = min(current + timedelta(minutes=SEGMENT_MIN), end)
    
    ts_str = current.strftime("%Y/%m/%d %H:%M:%S")
    te_str = seg_end.strftime("%Y/%m/%d %H:%M:%S")
    sol_file = f"{OUTPUT_DIR}/sol_{segment_idx:03d}.pos"
    
    print(f"Обработка [{ts_str} – {te_str}] → {sol_file}")
    
    cmd = [
        "/Users/sergeidolin/RTKLIB/app/consapp/rnx2rtkp/gcc/rnx2rtkp",
        "-k", CONFIG_FILE,
        "-o", sol_file,
        "-ts", *ts_str.split(),
        "-te", *te_str.split(),
        ROVER_RNX, NAV_FILE, SP3_FILE, CLK_FILE
    ]
    # cmd = [
    #     "/Users/sergeidolin/RTKLIB/app/consapp/rnx2rtkp/gcc/rnx2rtkp",
    #     "-k", CONFIG_FILE,
    #     "-o", sol_file,
    #     "-ts", *ts_str.split(),
    #     "-te", *te_str.split(),
    #     ROVER_RNX, BASE_FILE, NAV_FILE
    # ]
    
    try:
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError as e:
        print(f"  → ошибка: {e}")
    
    current = seg_end
    segment_idx += 1

print(f"\n✅ Готово: {segment_idx} сегментов обработано.")

Обработка [2025/07/14 10:12:32 – 2025/07/14 12:42:32] → solutions_all_AR/sol_000.pos
Обработка [2025/07/14 12:42:32 – 2025/07/14 15:12:32] → solutions_all_AR/sol_001.pos
Обработка [2025/07/14 15:12:32 – 2025/07/14 17:42:32] → solutions_all_AR/sol_002.pos
Обработка [2025/07/14 17:42:32 – 2025/07/14 20:12:32] → solutions_all_AR/sol_003.pos
Обработка [2025/07/14 20:12:32 – 2025/07/14 22:42:32] → solutions_all_AR/sol_004.pos
Обработка [2025/07/14 22:42:32 – 2025/07/15 01:12:32] → solutions_all_AR/sol_005.pos
Обработка [2025/07/15 01:12:32 – 2025/07/15 03:42:32] → solutions_all_AR/sol_006.pos
Обработка [2025/07/15 03:42:32 – 2025/07/15 06:12:32] → solutions_all_AR/sol_007.pos

✅ Готово: 8 сегментов обработано.
